In [ ]:
# import knihoven
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uncertainties import ufloat as uf
from scipy.optimize import curve_fit
from scipy import stats
from pathlib import Path

# path config
# cesta do Downloads funguje stejne na Windows i Linuxu
downloads_dir = Path.home() / "Downloads"

# ujistime se, ze slozka existuje (kdyby nahodou ne)
downloads_dir.mkdir(parents=True, exist_ok=True)

# ulozeni - Path se automaticky prevede na string se spravnymi lomitky
# output_path = downloads_dir / 'muj_graf.png'

In [ ]:
# definice funkci

# nejistoty typu B pro digitalni pristroje
def unc_B_digital(reading, percent_reading, digits, resolution):    # vse musi byt ve spravnych jednotkach; funguje pouze pro jedny podminky zaroven
    """
    Vypocet nejistoty typu B pro digitalni mereni (ponechame krajni)

    reading : float nebo array
        namerena hodnota
    percent_reading : float
        % of reading
    digits : float
        pocet digitu
    resolution : float
        nejmensi dilek
    """

    max_error = abs(reading) * (percent_reading/100) + digits * resolution  # max_error -- krajni nejistota
    u_B = max_error

    return u_B

# funkce na Studentuv koeficient
def StudCoef(confidence, dof): 
    """
    Parametry
    confidence : float
        hladina spolehlivosti, typicke hodnoty 0.683, 0.9973;
    dof : 
        pocet stupnu volnosti, pro jednoduchou statistiku array.size-1
    Returns : float
    Studentuv koeficient pro danou hladinu spolehlivosti a pocet stupnu volnosti
    """
    alpha = 1 - confidence
    return stats.t.ppf(1 - alpha/2, dof)

In [ ]:
# nacteni google tabulek
sheet_id = '1UsP814IJrfRsE0t9Es3ti1vHPey5zNOwflnBbbTUxkY'
gid = '312593068'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_impedance = pd.read_csv(url)

In [ ]:
# impedance


Ri = df_impedance['Ri'].iloc[0]
odpor_u20 = np.array(df_impedance['odpor_U20'])*1e-3
odpor_i = odpor_u20 / Ri
odpor_um0 = np.array(df_impedance['odpor_UM0'])*1e-3
odpor_rr = odpor_um0 / odpor_i
print(f'RR = {odpor_rr}')

kond_u20 = np.array(df_impedance['kond_U20'])*1e-3
kond_i = kond_u20 / Ri
kond_um0 = np.array(df_impedance['kond_UM0'])
kond_z = kond_um0 / kond_i
kond_f = np.array(df_impedance['kond_f'])
kond_phi = np.array(df_impedance['kond_phiM2'])*(np.pi/180)
kond_c = -1/(2*np.pi*kond_f*kond_z*np.sin(kond_phi))
kond_rc = kond_z * np.cos(kond_phi)
print(f'ZC = {kond_z},\nC  = {kond_c},\nRC = {kond_rc}')

civka_u20 = np.array(df_impedance['civka_U20'])*1e-3
civka_i = civka_u20 / Ri
civka_um0 = np.array(df_impedance['civka_UM0'])
civka_z = civka_um0 / civka_i
civka_f = np.array(df_impedance['civka_f'])
civka_phi = np.array(df_impedance['civka_phiM2'])*(np.pi/180)
civka_l = (civka_z*np.sin(civka_phi))/(2*np.pi*civka_f)
civka_rl = civka_z*np.cos(civka_phi)
print(f'ZL = {civka_z},\nL  = {civka_l},\nRL = {civka_rl}')

In [ ]:
# nacteni google tabulek
sheet_id = '1UsP814IJrfRsE0t9Es3ti1vHPey5zNOwflnBbbTUxkY'
gid = '800069381'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

df_rezonance = pd.read_csv(url)

In [ ]:
# rezonance
rez_f = np.array(df_rezonance['f'])
rez_um0 = np.array(df_rezonance['UM0'])*1e-3 # hodnoty vetsi nez 1000 byly namereny v rezimu 9.999 Volt
rez_u20 = np.array(df_rezonance['U20'])*1e-3
rez_phi = np.array(df_rezonance['phiM2'])*(np.pi/180)
rez_g = rez_u20/(Ri*rez_um0)
rez_phi_g = -rez_phi
print(f'G = {rez_g}\nphi_g = {rez_phi_g*180/np.pi}')


In [ ]:
# fit a graf vodivosti abs(G)

def model_G(f, R, L, C):
    return 1/((R**2 + (2*np.pi*f*L - 1/(2*np.pi*f*C))**2)**0.5)
p0_G = [30, 113*1e-3, 217*1e-9]
popt_G, pcov_G = curve_fit(model_G, rez_f, rez_g, p0_G)
R, L, C = popt_G
unc_R, unc_L, unc_C = np.sqrt(np.diag(pcov_G))
R = uf(R, unc_R)
L = uf(L, unc_L)
C = uf(C, unc_C)
F = 1/L
Q = (1/R)*((L/C)**0.5)
alpha = R/(2*L)
print(f'R = {R:.1u}\nL = {L:.1u}\nC = {C:.1u}\nF = {F:.1u}\nQ = {Q:.1u}\nalpha = {alpha:.1u}')

def model_phi(f, R, L, C):
    return np.atan(((1/(2*np.pi*f*C))-(2*np.pi*f*L))/(R))
p0_phi = [30, 113*1e-3, 217*1e-9]
popt_phi, pcov_phi = curve_fit(model_phi, rez_f, rez_phi_g, p0_phi)
#R, L, C = popt_phi
#unc_R, unc_L, unc_C = np.sqrt(np.diag(pcov_phi))
#print(f'R = {R}\nL = {L}\nC = {C}')

fig, ax = plt.subplots(2, 1, figsize=(9,16), sharex=True)
rez_x = 1013.82
#ax[0].set_xlabel(fr"$f\,[Hz]$", fontsize=20)
ax[0].set_ylabel(fr"$G\,[Sm]$", fontsize=20)
ax[0].scatter(rez_f, rez_g, marker='.', s=100, label='Hodnoty', color='blue')
linspace_f = np.linspace(min(rez_f), max(rez_f), 1000)
ax[0].plot(linspace_f, model_G(linspace_f, *popt_G), label="Fit", color='red')
ax[0].axvline(x=rez_x, color='k', linestyle='-.', linewidth=1.5, label='Maximum')
ax[0].tick_params(labelsize=15)
ax[0].grid(True, alpha=0.7)
ax[0].legend(fontsize=15)

ax[1].set_xlabel(fr"$f\,[Hz]$", fontsize=20)
ax[1].set_ylabel(fr"$\phi\,[deg]$", fontsize=20)
ax[1].scatter(rez_f, rez_phi_g*(180/np.pi), marker='.', s=100, label='Hodnoty', color='blue')
ax[1].plot(linspace_f, model_phi(linspace_f, *popt_phi)*(180/np.pi), label="Fit", color='red')
ax[1].axvline(x=rez_x, color='k', linestyle='-.', linewidth=1.5, label='Rezonance')
ax[1].tick_params(labelsize=15)
ax[1].grid(True, alpha=0.7)
ax[1].legend(fontsize=15)
fig.subplots_adjust(hspace=0.0)
path_vodivost = downloads_dir / 'vodivost.png'
plt.savefig(path_vodivost, dpi=300, bbox_inches='tight')


In [ ]:
# prechodovy jev
